In [1]:
import os
import sys
import torch
import transformers

sys.path.append(os.path.dirname(os.getcwd()))

from transformers import AutoTokenizer, GenerationConfig

from model.configuration_sophie0 import Sophie0Config
from model.modeling_sophie0 import Sophie0ForCausalLM

base_path = os.path.dirname(os.getcwd())
tokenizer_path = os.path.join(base_path, "model/tokenizer")
model_path = os.path.join(base_path, "result/pretrain/finish/pytorch_model.bin")

tokenizer: AutoTokenizer = AutoTokenizer.from_pretrained(tokenizer_path, use_fast=True, trust_remote_code=True, local_files_only=True)
model = Sophie0ForCausalLM(Sophie0Config())
model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=True))

/home/sophie/micromamba/envs/fla/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sophie/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:984: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/home/sophie/micromamba/envs/fla/lib/python3.12/site-packages/flash_attn/ops/triton/layer_norm.py:1043: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


<All keys matched successfully>

In [2]:
device = "cuda:0"
dtype = torch.bfloat16
model = model.to(device=device)

In [3]:
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=256,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.8,
    num_beams=1,
    repeat_penalty=1.2
)

prompt = [
    "<s><user>请问你是由谁训练研发的呢？</s>\n<s><bot>",
    "<s><user>能否解释一下Transformer架构呢？</s>\n<s><bot>",
    "<s><user>Could you please give a C++ example for quick sort?</s>\n<s><bot>",
    "<s><user>能否介绍一下中国的首都呢？</s>\n<s><bot>",
    "<s><user>Could you please tell me a joke about the weather?</s>\n<s><bot>"
]
input_ids = tokenizer(prompt, return_tensors="pt", padding="longest", padding_side="left").input_ids.to(device)

In [4]:
with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <s><user>请问你是由谁训练研发的呢？</s>
<s><bot>丨"哪咤之魔童降世"
在2014年之前,中国国内媒体就一直处于一个舆论漩涡之中.2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.官方媒体也迅速将这一消息迅速转发至全国.
  2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.
  2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.
  2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.
  2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.
  2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.
  2014年11月17日,中国官方媒体"瞭望"新闻发布后,一则消息迅速成为舆论焦点.
  2014年11月17日,中国官方
1: <pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>电影"梁山:三国风·西游记"剧照
  电影"梁山:三国风·西游记"剧照
  电影"梁山:三国风·西游记"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:三国风·Types"剧照
  电影"梁山:
2: <s><user>Could you please give a C++ example for quick sort?</s>
<s><bot>, Chinese, and the U.S. Army Posted By David Folkenflik On August 

In [5]:
# 使用beam search
generate_config = GenerationConfig(
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_new_tokens=256,
    do_sample=True,
    top_k=20,
    top_p=0.8,
    temperature=0.8,
    num_beams=3,
    num_return_sequences=1,
    repeat_penalty=1.2
)

with torch.amp.autocast(device_type="cuda", dtype=dtype):
    outputs = model.generate(input_ids, generate_config)

outputs = tokenizer.batch_decode(outputs, skip_special_tokens=False)
for i, output in enumerate(outputs):
    print(f"{i}: {output}")

0: <s><user>请问你是由谁训练研发的呢？</s>
<s><bot>: 20 years on
SlateFuture TenseThe citizen's guide to the future.March 25 2004 5:54 PM
By Michael Kinsley
You've probably heard the term "authenticity," but what do you actually mean by it? Take a look at these articles from several years ago. In the 1980s and 1990s, I wrote an article about the future of news -- the future of the news industry. In the 1990s, it was called "The New News," and it was called "The New News." In the 1990s, it was called "The New News." In the 1990s, it was called "The New News." In the 1990s, it was called "The World." In the 2000s, it was called "The World." In the 2000s, it was called "The World." In the 2000s, it was called "The World." In the 2000s, it was called "The World." In the 2000s, it
1: <pad><pad><s><user>能否解释一下Transformer架构呢？</s>
<s><bot>(1911年-1945年)是英国著名的数学家,物理学家和数学家. 1911年10月10日生于英国伦敦. 1911年10月10日生于英国伦敦. 1911年10月10日生于英国伦敦. 1911年10月10日生于英国伦敦. 1911年10月10日生于英国伦敦. 1911年10月10日生于英国伦敦. 1911年10月10日生于英国伦敦. 1911